# Домашнее задание: Откуда берутся датасеты?

Сбор датасета с сайта [nsk-kraeved.ru](https://nsk-kraeved.ru), TF-IDF преобразование текста и обучение модели линейной регрессии.

**TF-IDF** (Term Frequency — Inverse Document Frequency) — это статистическая мера, используемая в обработке естественного языка (NLP) для оценки важности слова в конкретном документе по отношению ко всей коллекции документов (корпусу).

**Источник данных:** [nsk-kraeved.ru](https://nsk-kraeved.ru) — краеведческий форум
Новосибирска. У каждой темы форума есть текст обсуждения (история
здания/улицы/моста и т.п.) и число просмотров — это и берём в качестве текстового
описания объекта и целевой переменной.


## Часть 1. Парсинг

### Устройство парсера (`scraper.py`)

Два этапа, оба ограничивают частоту запросов (`time.sleep(0.4)` между запросами)
и пишут результат в CSV (можно прерывать и продолжать):

1. **`stage1_collect_metadata`** — по списку отобранных подфорумов (переменная
   `FORUM_IDS`, ~27 разделов: улицы, площади, мосты, транспорт, районы, музеи, история и т.д.) обходит все страницы списка тем и собирает
   `topic_id`, `title`, `views`, `replies`.
2. **`stage2_collect_text`** — по каждой теме скачивает первую страницу
   (`viewtopic.php?id=...`), склеивает текст всех сообщений на ней
   (`div.post-content`) и берёт `data-posted` первого сообщения как дату создания
   темы.

Ниже — демонстрация работы парсера на одном подфоруме (без полного прогона,
он уже выполнен отдельно, см. `python scraper.py`, лог в `scrape.log`).


In [1]:
import scraper as s

demo_topics = s.collect_forum_topics(11, "Скульптурный город")
print(f"Собрано тем в тестовом подфоруме: {len(demo_topics)}")
demo_topics[:3]


Собрано тем в тестовом подфоруме: 378


[{'topic_id': '17913',
  'title': 'Мозаика на кирпичной стене ТП между Академическая, 13, 15, 19',
  'replies': 5,
  'views': 114,
  'forum_id': 11,
  'forum_name': 'Скульптурный город'},
 {'topic_id': '17914',
  'title': 'Кот учёный возле вокзала станции "Сеятель"',
  'replies': 5,
  'views': 101,
  'forum_id': 11,
  'forum_name': 'Скульптурный город'},
 {'topic_id': '17894',
  'title': 'Бюст Шарля де Голля в комнате приёма Vip гостей Дома Учёных СО РАН',
  'replies': 4,
  'views': 82,
  'forum_id': 11,
  'forum_name': 'Скульптурный город'}]

In [2]:
demo_html = s.fetch(f"{s.BASE_URL}/viewtopic.php?id={demo_topics[0]['topic_id']}")
demo_text, demo_created_ts = s.parse_topic_page(demo_html)
print("Дата создания (unix ts):", demo_created_ts)
print("Длина текста:", len(demo_text))
print(demo_text[:400])


Дата создания (unix ts): 1785741180
Длина текста: 281
. Отредактировано Евгений Козионов (Сегодня 09:08:14) Оставлю небольшую зацепку. Подсказка: В центре мозаики указано и имя автора. Что-то у меня не бьёт фото слишком старые, хотя берез не заметил да  и дома 13, 15, 19 сильно разбросаны, между ними затесался 17 Исправил. Извиняюсь.


### Загрузка полного собранного датасета

Полный сбор сохранён в `data/nsk_kraeved_topics.csv`.


In [3]:
import pandas as pd

df = pd.read_csv("data/nsk_kraeved_topics.csv")
print(df.shape)
df.head()


(14700, 9)


,forum_id,forum_name,topic_id,title,views,replies,text,created_ts,scraped_ts
0,51,Новониколаевск,17942,Фотографии... ?,138,4,... Отредактировано VECTOR (Вчера 22:21:56) Ч...,1786179451,1786263980
1,51,Новониколаевск,17902,Горюшкин Первые жители Новониколаевска(1893—18...,61,1,/// Отредактировано alippa (31-07-2026 14:54:4...,1785445772,1786263982
2,51,Новониколаевск,375,Военный городок,13365,170,. Городок на Тополёвой? Верно. Надпись на обр...,1254580933,1786263984
3,51,Новониколаевск,657,"ул.Чаплыгина, 53. Доходный дом.",1467,40,"По годам попал? В смысле - года постройки, кон...",1295629257,1786263986
4,51,Новониколаевск,5377,"Дом на Асинкритовской улице (ул. Чаплыгина, 75)",1469,18,??? Отредактировано d_popovskiy (29-10-2016 18...,1477739529,1786263988


In [4]:
df["forum_name"].value_counts()


forum_name
Новосибирск 60-х - 70-х                            1559
Новосибирск 40-х - 50-х                             732
Новосибирск 80-х - 90-х                             630
Книги и публикации                                  585
Просто поболтать                                    546
                                                   ... 
Агитплощадки советских времён                         4
Краснообск (ВАСХНИЛ)                                  3
Чердаки                                               2
Посты воздушного наблюдения, оповещения и связи       2
Первый герб Новосибирска                              2
Name: count, Length: 132, dtype: int64

## Часть 2. NLP

### Целевая переменная

Возраст темы в днях = (момент скачивания − дата создания темы). Целевая переменная —
среднее число просмотров в день. Так как просмотры очень скошены (несколько
популярных тем против массы рядовых), моделируем `log1p(views_per_day)`.


In [5]:
import numpy as np

df["created_ts"] = pd.to_numeric(df["created_ts"], errors="coerce")
df = df.dropna(subset=["created_ts", "text"]).copy()

df["age_days"] = (df["scraped_ts"] - df["created_ts"]) / 86400
df["age_days"] = df["age_days"].clip(lower=1)  # не даём совсем свежим темам взлетать до бесконечности

df["views_per_day"] = df["views"] / df["age_days"]
df["target"] = np.log1p(df["views_per_day"])

# выкидываем темы с совсем пустым/крошечным текстом - в них нет сигнала для tf-idf
df["text"] = df["text"].fillna("").astype(str)
df = df[df["text"].str.len() >= 30].reset_index(drop=True)

print(df.shape)
df[["views", "age_days", "views_per_day", "target"]].describe()


(14477, 12)


,views,age_days,views_per_day,target
count,14477.000000,14477.000000,14477.000000,14477.000000
mean,776.862955,2669.147634,0.504441,0.305819
std,1630.076375,1733.195771,1.859566,0.340056
min,0.000000,1.000000,0.000000,0.000000
25%,254.000000,1198.047199,0.119281,0.112686
50%,420.000000,2377.946493,0.225652,0.203473
75%,727.000000,4158.867269,0.439849,0.364538
max,40034.000000,6231.217095,138.000000,4.934474


### Очистка текста от служебных артефактов форума

Первый проход по данным показал утечку: движок форума автоматически добавляет к
каждому отредактированному сообщению приписку вида
`Отредактировано <Имя> (07-06-2026 21:45:20)`, а при цитировании — приписку
`<Имя> написал(а):`. Эти даты/имена — не часть рассказа об объекте, а форумная
разметка, которая при этом коррелирует с "свежестью" темы (а значит и с целевой
переменной). На полном датасете (все 132 подфорума) обнаружилась и вторая
утечка того же типа: часть тем содержит ссылки на Google Maps
(`https://maps.app.goo.gl/...`), и обрывки этих ссылок ("https", "app goo",
"goo gl") оказались в топе значимых "слов" — это чистый URL-мусор, а не текст
про объект. Убираем ссылки и служебные пометки, а токенайзер TF-IDF настраиваем
так, чтобы он не превращал даты/номера в отдельные "слова" — токеном считается
только последовательность из букв.


In [6]:
import re

URL_RE = re.compile(r"(https?://\S+)|(\bwww\.\S+)", re.UNICODE)
EDIT_NOTICE_RE = re.compile(r"Отредактировано[^)]*\)", re.UNICODE)
# "<ник> написал(а):" - форумная приписка при цитировании, ник может быть
# кириллическим или латинским, в любом регистре (никнеймы часто латиницей).
QUOTE_ATTR_RE = re.compile(r"[^\s]+(?:\s+[^\s]+){0,2}\s*написал\(а\):", re.UNICODE)


def clean_text(text):
    text = URL_RE.sub(" ", text)
    text = EDIT_NOTICE_RE.sub(" ", text)
    text = QUOTE_ATTR_RE.sub(" ", text)
    return text


df["text"] = df["text"].map(clean_text)
df = df[df["text"].str.len() >= 30].reset_index(drop=True)
print(df.shape)


(14388, 12)


### Train/test split (75% / 25%)

In [7]:
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"], df["target"], test_size=0.25, random_state=42
)
print(f"train: {len(X_train_text)}, test: {len(X_test_text)}")


train: 10791, test: 3597


### TF-IDF

* униграммы + биграммы (`ngram_range=(1, 2)`);
* русские стоп-слова (nltk);
* отсекаем слишком редкие (`min_df`) и слишком частые (`max_df`) токены;
* `token_pattern` — токен должен состоять из букв (не цифр), чтобы даты/номера
  домов/года не попадали в словарь как "слова";
* `norm=None` — отключаем L2-нормировку строк, включённую по умолчанию, как того
  требует задание.


In [8]:
import nltk

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

russian_stopwords = stopwords.words("russian")

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    stop_words=russian_stopwords,
    min_df=10,
    max_df=0.6,
    norm=None,
    token_pattern=r"(?u)\b[а-яёa-z]{2,}\b",
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)
print(X_train.shape, X_test.shape)


(10791, 31667) (3597, 31667)


### Линейная регрессия с регуляризацией

Целевая переменная непрерывна (log-просмотры/день) → гребневая регрессия (Ridge).
Параметр регуляризации подбираем кросс-валидацией. На таком объёме данных
(~11 тыс. документов, десятки тысяч признаков) стандартный `RidgeCV` с
solver'ом по умолчанию (`sparse_cg`) считается очень долго — берём `Ridge`
с solver'ом `lsqr` (быстрее сходится на разреженных матрицах) через
`GridSearchCV` и считаем фолды параллельно (`n_jobs=-1`).


In [9]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

alphas = np.logspace(-2, 6, 17)
grid = GridSearchCV(
    Ridge(solver="lsqr"),
    param_grid={"alpha": alphas},
    scoring="r2",
    cv=5,
    n_jobs=-1,
)
grid.fit(X_train, y_train)
model = grid.best_estimator_
print("Лучшее alpha:", grid.best_params_["alpha"])


Лучшее alpha: 31622.776601683796


In [10]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print(f"R^2   = {r2:.3f}")
print(f"MAE   = {mae:.3f}  (в шкале log1p(просмотров/день))")
print(f"RMSE  = {rmse:.3f}  (в шкале log1p(просмотров/день))")


R^2   = 0.211
MAE   = 0.170  (в шкале log1p(просмотров/день))
RMSE  = 0.289  (в шкале log1p(просмотров/день))


### Визуализация коэффициентов (топ-25 положительных + топ-25 отрицательных = 50)

Коэффициент регрессии показывает, как наличие слова/биграммы связано с
логарифмом просмотров/день: положительный — слово встречается в темах с
большим числом просмотров/день, отрицательный — с меньшим.


In [11]:
import matplotlib.pyplot as plt

feature_names = np.array(vectorizer.get_feature_names_out())
coefs = model.coef_

top_pos_idx = np.argsort(coefs)[-25:]
top_neg_idx = np.argsort(coefs)[:25]
top_idx = np.concatenate([top_neg_idx, top_pos_idx])

plot_words = feature_names[top_idx]
plot_coefs = coefs[top_idx]
order = np.argsort(plot_coefs)
plot_words = plot_words[order]
plot_coefs = plot_coefs[order]

POSITIVE = "#2a78d6"   # blue - больше просмотров/день
NEGATIVE = "#e34948"   # red  - меньше просмотров/день
GRID = "#e1e0d9"
TEXT_PRIMARY = "#0b0b0b"
TEXT_MUTED = "#898781"

colors = [POSITIVE if c >= 0 else NEGATIVE for c in plot_coefs]

fig, ax = plt.subplots(figsize=(9, 14), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")
y_positions = np.arange(len(plot_words))
ax.barh(y_positions, plot_coefs, color=colors, height=0.65)

ax.set_yticks(y_positions)
ax.set_yticklabels(plot_words, fontsize=9, color=TEXT_PRIMARY)
ax.axvline(0, color=TEXT_MUTED, linewidth=1)
ax.set_xlabel("Коэффициент регрессии (влияние на log1p(просмотров/день))", color=TEXT_PRIMARY)
ax.set_title("Топ-50 слов/биграмм по влиянию на просматриваемость темы", color=TEXT_PRIMARY, fontsize=13)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRID)
ax.tick_params(axis="y", length=0)
ax.tick_params(axis="x", colors=TEXT_MUTED)
ax.xaxis.grid(True, color=GRID, linewidth=1)
ax.set_axisbelow(True)

legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color=POSITIVE, label="Больше просмотров/день"),
    plt.Rectangle((0, 0), 1, 1, color=NEGATIVE, label="Меньше просмотров/день"),
]
ax.legend(handles=legend_handles, loc="lower right", frameon=False, fontsize=9, labelcolor=TEXT_PRIMARY)

plt.tight_layout()
plt.savefig("top50_coefficients.png", dpi=150, facecolor="#fcfcfb")
plt.show()


<Figure size 900x1400 with 1 Axes>

### Интерпретация

Датасет здесь — полный сайт (все 132 подфорума, ~14.7 тыс. тем), а не выборка
из 27 отобранных вручную разделов, как в первой версии этого разбора. Полный
охват принёс не только больше данных, но и новые виды утечки, которые
пришлось находить итеративно, тем же способом, что и раньше — смотреть на
топ-слова и спрашивать "это вообще про объект или про формат форума?":

1. **Даты автоправок** ("Отредактировано `<имя>` (дата)") — та же утечка,
   что и в версии на 27 подфорумах: коррелирует со "свежестью" темы, но не с
   её содержанием. Убирается тем же регулярным выражением.
2. **Ссылки на Google Maps** — новая находка на полном датасете: в некоторых
   темах ссылки вида `https://maps.app.goo.gl/...` разбивались токенайзером на
   осколки "https", "app goo", "goo gl", которые заняли верхушку топ-50 (при
   них R² на тесте был ≈ 0.22, но это в основном не сигнал о содержании, а
   утечка через технический формат ссылки). После того как ссылки стали
   вырезаться целиком регулярным выражением, R² просел лишь незначительно —
   с ≈0.22 до **≈0.21** (см. фактическое значение в выводе ячейки выше) — то
   есть на этот раз оценка честная не только благодаря чистке, но и потому что
   большая часть прежнего R² была настоящим сигналом, не утечкой.

**Слова с положительным коэффициентом** (синие бары, темы с ними в среднем
просматриваются чаще) — узнаваемая топонимика и типы объектов: *краевед,
кладбища, табличка, бердское (шоссе), академгородке/академгородка,
космоснимке, плиты*. Академгородок и Бердское шоссе — крупные, всем известные
районы/трассы города, поэтому логично, что темы про них собирают больше
просмотров в день, чем темы про рядовые дворы и здания.

**Слова с отрицательным коэффициентом** (красные бары, темы с ними в среднем
просматриваются реже) — снова узкоспециализированные и служебные сюжеты:
*дача, революции, машины, железной (дороги), промышленности, постройки, скан*
(отсылка на скан документа/газеты — нишевый архивный материал), а также
формальные слова самого форума-"фотозагадки": *вопрос, новая фотозагадка,
таки* — они говорят о жанре обсуждения ("угадайте объект"), а не о самом
объекте, как отмечено и в `topic_modeling.ipynb`. Отдельные личные имена
(*женя, андрей, костя*) — остаточный след конкретных активных участников,
не до конца устранённый очисткой цитат.

**Общий вывод.** На полном сайте регрессия объясняет заметно больше дисперсии,
чем на кураторской подвыборке из 27 разделов (там R² после честной чистки был
≈ 0.09) — больше данных и более разнообразные подфорумы (в том числе бытовые,
не только исторические) дали более устойчивую и интерпретируемую модель.
Тем не менее и здесь показатель умеренный: просматриваемость темы определяется
не только текстом, но и её возрастом, активностью конкретных участников и
попаданием в "горячие" темы форума.
